# Experiment SQ3 — Raw IQ vs STFT Spectrogram Comparison
**Thesis:** Lightweight Device Authentication in Wireless Communication Using RF Fingerprinting  
**Authors:** Amitha · Tharangi Madushani  
**Supervisor:** Prof. Qinghua Wang

---

## Research Gap G3

Yan et al. [5] explicitly note the absence of a controlled head-to-head comparison between
raw IQ and STFT spectrogram inputs using the same CNN architecture family and same SNR sweep.
This notebook performs exactly that experiment.

| Model | Input | Shape | Architecture |
|---|---|---|---|
| 1D CNN | Raw IQ | (128, 2) | Conv1D × 3 + GAP |
| 2D CNN | STFT magnitude | (33, 17, 1) | Conv2D × 3 + GAP |

**Data:** BPSK · S1+S2 · clean+SNR10+SNR0 training · evaluate at clean/SNR20/SNR10/SNR0  
**Metrics:** Accuracy (%), FAR (%), Inference latency (ms/window)

## Section 1 — Mount Drive + Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import signal as scipy_signal
from sklearn.metrics import confusion_matrix, classification_report
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import gc

BASE         = '/content/drive/MyDrive/My Thesis'
PREPROCESSED = os.path.join(BASE, 'Preprocessed')
MODEL_DIR    = os.path.join(BASE, 'Models', 'SQ3')
os.makedirs(MODEL_DIR, exist_ok=True)

# Fixed config — BPSK only for fair comparison with Experiment A
MODULATION   = 'BPSK'
SESSIONS     = ['S1', 'S2']
AUG_TAGS     = ['clean', 'SNR10', 'SNR0']   # training SNR tags
EVAL_SNRS    = [None, 20, 10, 0]             # None = clean
BATCH_SIZE   = 256
MAX_EPOCHS   = 60
LR           = 1e-4
RANDOM_SEED  = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# STFT parameters — tuned for 128-sample windows
STFT_NPERSEG = 32
STFT_NOVERLAP = 24
STFT_NFFT    = 64
# Output shape: (33 freq bins, 17 time steps, 1 channel)

sq3_results = {}

print(f'BASE     : {os.listdir(BASE)}')
print(f'Prep S1  : {os.listdir(os.path.join(PREPROCESSED, "S1"))[:4]} ...')
print(f'Model dir: {MODEL_DIR}')
print(f'TF       : {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
print(f'GPU      : {gpus[0].name if gpus else "CPU only"}')

## Section 2 — Data Loading + STFT Extraction Helpers

In [ ]:
def load_npy(session, modulation, snr_tag):
    folder = os.path.join(PREPROCESSED, session)
    X = np.load(os.path.join(folder, f'{modulation}_{snr_tag}_X.npy'))
    y = np.load(os.path.join(folder, f'{modulation}_{snr_tag}_y.npy'))
    return X, y


def load_bpsk_augmented():
    """Load BPSK data from S1+S2, clean+SNR10+SNR0 — same as Experiment A Model 3."""
    parts_X, parts_y = [], []
    for sess in SESSIONS:
        for tag in AUG_TAGS:
            X, y = load_npy(sess, MODULATION, tag)
            parts_X.append(X)
            parts_y.append(y)
            print(f'  Loaded {sess}/{tag}: {X.shape}')
            del X, y
    X_all = np.concatenate(parts_X)
    y_all = np.concatenate(parts_y)
    del parts_X, parts_y
    gc.collect()
    print(f'  Total: {X_all.shape}, labels: {np.bincount(y_all.astype(int))}')
    return X_all, y_all


def load_bpsk_eval(snr_tag):
    """Load BPSK eval data from S2 only (held-out session)."""
    X, y = load_npy('S2', MODULATION, snr_tag)
    return X, y


def iq_to_stft(X_iq):
    """
    Convert raw IQ windows to STFT magnitude spectrograms.
    Input:  X_iq  shape (N, 128, 2)
    Output: X_stft shape (N, 33, 17, 1)
    """
    N = X_iq.shape[0]
    spectrograms = []
    for i in range(N):
        # Treat IQ as complex signal
        iq_complex = X_iq[i, :, 0] + 1j * X_iq[i, :, 1]
        f, t, Zxx = scipy_signal.stft(
            iq_complex,
            nperseg=STFT_NPERSEG,
            noverlap=STFT_NOVERLAP,
            nfft=STFT_NFFT
        )
        mag = np.abs(Zxx)  # (33, 17)
        spectrograms.append(mag)
    X_stft = np.array(spectrograms)[..., np.newaxis]  # (N, 33, 17, 1)
    return X_stft.astype(np.float32)


# Verify STFT shape on a small batch
X_test, _ = load_npy('S1', MODULATION, 'clean')
X_stft_test = iq_to_stft(X_test[:10])
print(f'Raw IQ shape : {X_test[:10].shape}')
print(f'STFT shape   : {X_stft_test.shape}')
del X_test, X_stft_test
gc.collect()
print('Helpers ready.')

## Section 3 — Model Definitions

In [ ]:
def build_1d_cnn(input_shape=(128, 2), n_classes=2):
    """
    Identical to Experiment A architecture.
    Conv1D x3 + AveragePooling + GlobalAvgPool + Dense64 + Softmax
    """
    inp = keras.Input(shape=input_shape)
    x = layers.Conv1D(32, 7, padding='same', activation='relu')(inp)
    x = layers.AveragePooling1D(2)(x)
    x = layers.Conv1D(64, 5, padding='same', activation='relu')(x)
    x = layers.AveragePooling1D(2)(x)
    x = layers.Conv1D(128, 3, padding='same', activation='relu')(x)
    x = layers.AveragePooling1D(2)(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(n_classes, activation='softmax')(x)
    model = keras.Model(inp, out, name='1D_CNN_RawIQ')
    return model


def build_2d_cnn(input_shape=(33, 17, 1), n_classes=2):
    """
    2D CNN for STFT spectrogram input.
    Conv2D x3 + AveragePooling + GlobalAvgPool + Dense64 + Softmax
    Same design philosophy as 1D CNN — no BatchNorm, AveragePool, GlobalAvgPool.
    """
    inp = keras.Input(shape=input_shape)
    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(inp)
    x = layers.AveragePooling2D((2, 2))(x)
    x = layers.Conv2D(64, (3, 3), padding='same', activation='relu')(x)
    x = layers.AveragePooling2D((2, 2))(x)
    x = layers.Conv2D(128, (3, 3), padding='same', activation='relu')(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.5)(x)
    out = layers.Dense(n_classes, activation='softmax')(x)
    model = keras.Model(inp, out, name='2D_CNN_STFT')
    return model


# Build and summarise both
m1 = build_1d_cnn()
m2 = build_2d_cnn()
print('=== 1D CNN (Raw IQ) ===')
m1.summary()
print(f'\nParameters: {m1.count_params():,}')
print('\n=== 2D CNN (STFT) ===')
m2.summary()
print(f'\nParameters: {m2.count_params():,}')
del m1, m2
gc.collect()

## Section 4 — Load and Prepare Training Data

In [ ]:
from sklearn.model_selection import train_test_split

print('Loading BPSK augmented training data...')
X_iq_all, y_all = load_bpsk_augmented()

# Train/val split (80/20) — temporal split within augmented pool
X_iq_train, X_iq_val, y_train, y_val = train_test_split(
    X_iq_all, y_all, test_size=0.2,
    random_state=RANDOM_SEED, stratify=y_all
)

print(f'\nRaw IQ — Train: {X_iq_train.shape}, Val: {X_iq_val.shape}')
print(f'Label balance — Train: {np.bincount(y_train.astype(int))}, '
      f'Val: {np.bincount(y_val.astype(int))}')

# Convert to STFT for 2D CNN
print('\nGenerating STFT spectrograms (train)...')
X_stft_train = iq_to_stft(X_iq_train)
print(f'STFT train shape: {X_stft_train.shape}')

print('Generating STFT spectrograms (val)...')
X_stft_val = iq_to_stft(X_iq_val)
print(f'STFT val shape: {X_stft_val.shape}')

print('\nData preparation complete.')

## Section 5 — Train 1D CNN (Raw IQ)

In [ ]:
def get_callbacks(ckpt_path):
    return [
        EarlyStopping(monitor='val_loss', patience=8,
                      restore_best_weights=True, verbose=1),
        ModelCheckpoint(ckpt_path, monitor='val_loss',
                        save_best_only=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=4, min_lr=1e-6, verbose=1)
    ]


print('Training 1D CNN on raw IQ...')
model_1d = build_1d_cnn()
model_1d.compile(
    optimizer=keras.optimizers.Adam(LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

ckpt_1d = os.path.join(MODEL_DIR, 'sq3_1d_cnn.keras')
history_1d = model_1d.fit(
    X_iq_train, y_train,
    validation_data=(X_iq_val, y_val),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=get_callbacks(ckpt_1d),
    verbose=1
)

print(f'\n1D CNN training complete.')
print(f'Best val accuracy: {max(history_1d.history["val_accuracy"]):.4f}')
print(f'Saved: {ckpt_1d}')

## Section 6 — Train 2D CNN (STFT Spectrogram)

In [ ]:
print('Training 2D CNN on STFT spectrograms...')
model_2d = build_2d_cnn()
model_2d.compile(
    optimizer=keras.optimizers.Adam(LR),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

ckpt_2d = os.path.join(MODEL_DIR, 'sq3_2d_cnn.keras')
history_2d = model_2d.fit(
    X_stft_train, y_train,
    validation_data=(X_stft_val, y_val),
    epochs=MAX_EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=get_callbacks(ckpt_2d),
    verbose=1
)

print(f'\n2D CNN training complete.')
print(f'Best val accuracy: {max(history_2d.history["val_accuracy"]):.4f}')
print(f'Saved: {ckpt_2d}')

## Section 7 — Accuracy Evaluation per SNR

Evaluate both models on S2 held-out data at each SNR level.
S2 was never seen during training — honest cross-session evaluation.

In [ ]:
def snr_tag_from_level(snr_level):
    if snr_level is None:
        return 'clean'
    return f'SNR{snr_level}'


def evaluate_at_snr(model, snr_level, use_stft=False):
    tag = snr_tag_from_level(snr_level)
    try:
        X, y = load_npy('S2', MODULATION, tag)
    except FileNotFoundError:
        return None, None, None

    if use_stft:
        X = iq_to_stft(X)

    y_pred = np.argmax(model.predict(X, batch_size=512, verbose=0), axis=1)

    accuracy = np.mean(y_pred == y) * 100

    # FAR: DEV02 (label 1) incorrectly accepted as DEV01 (label 0)
    dev02_mask = (y == 1)
    far = np.mean(y_pred[dev02_mask] == 0) * 100 if dev02_mask.sum() > 0 else 0.0

    # FRR: DEV01 (label 0) incorrectly rejected
    dev01_mask = (y == 0)
    frr = np.mean(y_pred[dev01_mask] == 1) * 100 if dev01_mask.sum() > 0 else 0.0

    del X, y, y_pred
    gc.collect()
    return accuracy, far, frr


print('Evaluating 1D CNN (Raw IQ)...')
results_1d = {}
for snr in EVAL_SNRS:
    label = 'Clean' if snr is None else f'SNR{snr}'
    acc, far, frr = evaluate_at_snr(model_1d, snr, use_stft=False)
    results_1d[label] = {'accuracy': acc, 'far': far, 'frr': frr}
    print(f'  {label:10s}: Acc={acc:.2f}%  FAR={far:.2f}%  FRR={frr:.2f}%')

print('\nEvaluating 2D CNN (STFT)...')
results_2d = {}
for snr in EVAL_SNRS:
    label = 'Clean' if snr is None else f'SNR{snr}'
    acc, far, frr = evaluate_at_snr(model_2d, snr, use_stft=True)
    results_2d[label] = {'accuracy': acc, 'far': far, 'frr': frr}
    print(f'  {label:10s}: Acc={acc:.2f}%  FAR={far:.2f}%  FRR={frr:.2f}%')

sq3_results['accuracy_1d'] = results_1d
sq3_results['accuracy_2d'] = results_2d

## Section 8 — Inference Latency Measurement

Measure median latency per window for both models.
PR-1 requirement: < 10 ms per window.

In [ ]:
def measure_latency(model, sample_input, n_runs=1000):
    """
    Measure per-window inference latency.
    Warm-up 50 runs, then time 1000 sequential single-window predictions.
    """
    single = sample_input[:1]  # single window

    # Warm-up
    for _ in range(50):
        _ = model.predict(single, verbose=0)

    # Timed runs
    latencies = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        model.predict(single, verbose=0)
        latencies.append((time.perf_counter() - t0) * 1000)  # ms

    latencies = np.array(latencies)
    return {
        'median_ms': float(np.median(latencies)),
        'p95_ms':    float(np.percentile(latencies, 95)),
        'mean_ms':   float(np.mean(latencies)),
    }


# Load a small sample for timing
X_time_iq, _ = load_npy('S2', MODULATION, 'clean')
X_time_stft  = iq_to_stft(X_time_iq[:100])
X_time_iq    = X_time_iq[:100]

print('Measuring 1D CNN latency (1000 single-window predictions)...')
lat_1d = measure_latency(model_1d, X_time_iq)
print(f'  Median: {lat_1d["median_ms"]:.3f} ms  |  P95: {lat_1d["p95_ms"]:.3f} ms')

print('Measuring 2D CNN latency (1000 single-window predictions)...')
lat_2d = measure_latency(model_2d, X_time_stft)
print(f'  Median: {lat_2d["median_ms"]:.3f} ms  |  P95: {lat_2d["p95_ms"]:.3f} ms')

sq3_results['latency_1d'] = lat_1d
sq3_results['latency_2d'] = lat_2d
sq3_results['params_1d']  = int(model_1d.count_params())
sq3_results['params_2d']  = int(model_2d.count_params())

print(f'\nParameters — 1D CNN: {model_1d.count_params():,} | 2D CNN: {model_2d.count_params():,}')
pr1_1d = 'PASS' if lat_1d['median_ms'] < 10 else 'FAIL'
pr1_2d = 'PASS' if lat_2d['median_ms'] < 10 else 'FAIL'
print(f'PR-1 (< 10ms) — 1D CNN: {pr1_1d} | 2D CNN: {pr1_2d}')

del X_time_iq, X_time_stft
gc.collect()

## Section 9 — Plots: Training Curves + Accuracy Comparison + Latency

In [ ]:
fig = plt.figure(figsize=(20, 14))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

snr_labels  = ['Clean (∞ dB)', 'SNR 20 dB', 'SNR 10 dB', 'SNR 0 dB']
result_keys = ['Clean', 'SNR20', 'SNR10', 'SNR0']
acc_1d = [results_1d[k]['accuracy'] for k in result_keys]
acc_2d = [results_2d[k]['accuracy'] for k in result_keys]
far_1d = [results_1d[k]['far'] for k in result_keys]
far_2d = [results_2d[k]['far'] for k in result_keys]

# ── Plot 1: 1D CNN training curves
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(history_1d.history['accuracy'],     label='Train Acc')
ax1.plot(history_1d.history['val_accuracy'], label='Val Acc')
ax1.set_title('1D CNN Training (Raw IQ)', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_ylim([0.4, 1.05])

# ── Plot 2: 2D CNN training curves
ax2 = fig.add_subplot(gs[0, 1])
ax2.plot(history_2d.history['accuracy'],     label='Train Acc', color='orange')
ax2.plot(history_2d.history['val_accuracy'], label='Val Acc',   color='red')
ax2.set_title('2D CNN Training (STFT)', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.legend()
ax2.grid(True, alpha=0.3)
ax2.set_ylim([0.4, 1.05])

# ── Plot 3: Accuracy vs SNR comparison
ax3 = fig.add_subplot(gs[0, 2])
x = np.arange(len(snr_labels))
w = 0.35
bars1 = ax3.bar(x - w/2, acc_1d, w, label='1D CNN (Raw IQ)', color='steelblue')
bars2 = ax3.bar(x + w/2, acc_2d, w, label='2D CNN (STFT)',   color='darkorange')
ax3.set_title('Accuracy vs SNR\n(SQ3 Head-to-Head)', fontweight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels(snr_labels, rotation=15, ha='right', fontsize=9)
ax3.set_ylabel('Accuracy (%)')
ax3.set_ylim([0, 110])
ax3.axhline(95, color='green',  linestyle='--', alpha=0.7, label='SR-1: 95%')
ax3.axhline(80, color='purple', linestyle='--', alpha=0.7, label='PR-3: 80%')
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.3, axis='y')
for bar in bars1:
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
             f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=7)
for bar in bars2:
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
             f'{bar.get_height():.1f}', ha='center', va='bottom', fontsize=7)

# ── Plot 4: FAR vs SNR comparison
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(snr_labels, far_1d, 'o-', color='steelblue',   label='1D CNN (Raw IQ)', linewidth=2)
ax4.plot(snr_labels, far_2d, 's-', color='darkorange',  label='2D CNN (STFT)',   linewidth=2)
ax4.axhline(5, color='red', linestyle='--', alpha=0.7, label='SR-2: FAR ≤ 5%')
ax4.set_title('False Acceptance Rate vs SNR', fontweight='bold')
ax4.set_xlabel('SNR Condition')
ax4.set_ylabel('FAR (%)')
ax4.legend(fontsize=9)
ax4.grid(True, alpha=0.3)
ax4.tick_params(axis='x', rotation=15)

# ── Plot 5: Latency comparison
ax5 = fig.add_subplot(gs[1, 1])
models_lat = ['1D CNN\n(Raw IQ)', '2D CNN\n(STFT)']
medians = [lat_1d['median_ms'], lat_2d['median_ms']]
p95s    = [lat_1d['p95_ms'],    lat_2d['p95_ms']]
bars = ax5.bar(models_lat, medians, color=['steelblue', 'darkorange'], width=0.4)
ax5.errorbar(models_lat, medians,
             yerr=[np.array(p95s) - np.array(medians)],
             fmt='none', color='black', capsize=5)
ax5.axhline(10, color='red', linestyle='--', alpha=0.7, label='PR-1: 10 ms limit')
ax5.set_title('Inference Latency\n(Median + P95 error bar)', fontweight='bold')
ax5.set_ylabel('Latency (ms)')
ax5.legend(fontsize=9)
ax5.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, medians):
    ax5.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.05,
             f'{val:.2f} ms', ha='center', va='bottom', fontweight='bold')

# ── Plot 6: Summary comparison table
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')
table_data = [
    ['Metric', '1D CNN\n(Raw IQ)', '2D CNN\n(STFT)'],
    ['Parameters', f'{model_1d.count_params():,}', f'{model_2d.count_params():,}'],
    ['Latency (median)', f'{lat_1d["median_ms"]:.2f} ms', f'{lat_2d["median_ms"]:.2f} ms'],
    ['Acc @ Clean', f'{acc_1d[0]:.1f}%', f'{acc_2d[0]:.1f}%'],
    ['Acc @ SNR20', f'{acc_1d[1]:.1f}%', f'{acc_2d[1]:.1f}%'],
    ['Acc @ SNR10', f'{acc_1d[2]:.1f}%', f'{acc_2d[2]:.1f}%'],
    ['Acc @ SNR0',  f'{acc_1d[3]:.1f}%', f'{acc_2d[3]:.1f}%'],
    ['FAR @ Clean', f'{far_1d[0]:.1f}%', f'{far_2d[0]:.1f}%'],
    ['PR-1 (< 10ms)', 'PASS' if lat_1d['median_ms']<10 else 'FAIL',
                      'PASS' if lat_2d['median_ms']<10 else 'FAIL'],
    ['PR-2 (< 500K)', 'PASS' if model_1d.count_params()<500000 else 'FAIL',
                      'PASS' if model_2d.count_params()<500000 else 'FAIL'],
]
tbl = ax6.table(cellText=table_data[1:], colLabels=table_data[0],
                loc='center', cellLoc='center')
tbl.auto_set_font_size(False)
tbl.set_fontsize(9)
tbl.scale(1.2, 1.6)
ax6.set_title('SQ3 Summary Table', fontweight='bold', pad=10)

fig.suptitle('SQ3 — Raw IQ vs STFT Spectrogram: Head-to-Head Comparison\n'
             'BPSK · Multi-session (S1+S2) · Noise-augmented training',
             fontsize=13, fontweight='bold')

plot_path = os.path.join(MODEL_DIR, 'SQ3_comparison.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {plot_path}')

## Section 10 — Print Summary + Save Results

In [ ]:
def make_serializable(obj):
    if isinstance(obj, (np.float32, np.float64)):
        return float(obj)
    if isinstance(obj, (np.int32, np.int64)):
        return int(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    raise TypeError(f'Not serializable: {type(obj)}')


print('='*65)
print('SQ3 FINAL SUMMARY — Raw IQ vs STFT Spectrogram')
print('='*65)
print(f'{'Metric':<30} {'1D CNN (Raw IQ)':>15} {'2D CNN (STFT)':>15}')
print('-'*65)
print(f'{'Parameters':<30} {model_1d.count_params():>15,} {model_2d.count_params():>15,}')
print(f'{'Latency median (ms)':<30} {lat_1d["median_ms"]:>15.3f} {lat_2d["median_ms"]:>15.3f}')
print(f'{'Latency P95 (ms)':<30} {lat_1d["p95_ms"]:>15.3f} {lat_2d["p95_ms"]:>15.3f}')
print('-'*65)
for k, label in zip(result_keys, snr_labels):
    a1 = results_1d[k]['accuracy']
    a2 = results_2d[k]['accuracy']
    print(f'  Acc @ {label:<22} {a1:>15.2f}% {a2:>15.2f}%')
print('-'*65)
for k, label in zip(result_keys, snr_labels):
    f1 = results_1d[k]['far']
    f2 = results_2d[k]['far']
    print(f'  FAR @ {label:<22} {f1:>15.2f}% {f2:>15.2f}%')
print('='*65)

# Recommendation
winner_acc = '1D CNN' if acc_1d[2] >= acc_2d[2] else '2D CNN'
winner_lat = '1D CNN' if lat_1d['median_ms'] <= lat_2d['median_ms'] else '2D CNN'
print(f'\nHigher accuracy at SNR10: {winner_acc}')
print(f'Lower latency:            {winner_lat}')

# Save JSON
json_path = os.path.join(MODEL_DIR, 'sq3_results.json')
with open(json_path, 'w') as f:
    json.dump(sq3_results, f, indent=2, default=make_serializable)
print(f'\nResults saved: {json_path}')
print('SQ3 complete.')